# Mansam Llama QLoRA — Google Colab

Draft training workflow. No model has been trained yet. Select **Runtime → Change runtime type → GPU**. Review the dataset before training; it currently covers evidence-grounded prices and notes, not full conversation behavior. You need approved access to the Llama model weights. Use a private notebook and do not publish credentials.

In [ ]:
!nvidia-smi
!pip -q install "transformers==4.56.2" "trl==0.23.1" "peft==0.17.1" "datasets==4.0.0" "accelerate==1.10.1" "bitsandbytes==0.47.0"

## Upload the prepared package
Upload `mansam-colab-training.zip`. Only the prepared fine-tuning files are needed; do not upload your whole website or tokens.

In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile
uploaded = files.upload()
archive_name = next(name for name in uploaded if name.endswith(".zip"))
base = Path("/content/mansam").resolve()
base.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[archive_name])) as archive:
    for member in archive.infolist():
        target = (base / member.filename).resolve()
        if not target.is_relative_to(base):
            raise ValueError("Unsafe archive path")
    archive.extractall(base)
print("Package extracted")

## Authenticate privately
Use a token authorized to read the gated Llama repository after accepting its model terms. The input is hidden and the token is only kept in this runtime environment.

In [ ]:
import os
from getpass import getpass
os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")
from huggingface_hub import hf_hub_download
hf_hub_download("meta-llama/Llama-3.1-8B-Instruct", "config.json", token=os.environ["HF_TOKEN"])
print("Model access confirmed")

## Review examples before training
Train/validation product-name groups do not overlap. Template-based validation loss does not prove improved customer conversations.

In [ ]:
import json
root = base / "finetuning"
print((root / "data/report.json").read_text())
rows = [json.loads(line) for line in (root / "data/train.jsonl").read_text().splitlines()]
for row in rows[:2]:
    print(json.dumps(row, ensure_ascii=False, indent=2))

## GPU smoke test
This downloads model weights and trains for two steps. If GPU memory is exhausted, use a larger GPU or reduce sequence length; do not assume the full job will fit.

In [ ]:
os.environ["MANSAM_TRAIN_STEPS"] = "2"
!python /content/mansam/finetuning/train_qlora.py

## One-epoch training
Run only after reviewing the examples and completing the smoke test successfully. This reloads the base model and starts a fresh one-epoch run. For production behavior, first add and evaluate multi-turn examples covering interruptions and preference changes.

In [ ]:
os.environ.pop("MANSAM_TRAIN_STEPS", None)
!python /content/mansam/finetuning/train_qlora.py

## Export adapter
This saves adapter weights, not a standalone full model. The public Novita endpoint will not automatically load this adapter. Deployment needs a service that loads the base model plus adapter; evaluate English/Arabic answers and product grounding before switching your chatbot.

In [ ]:
import shutil
archive = shutil.make_archive("/content/Llama-Mansam-adapter", "zip", root / "Llama-Mansam-adapter")
files.download(archive)